In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(r'/content/Diseases_and_Symptoms_two_columns.csv')
df.shape

(246945, 2)

In [ ]:
df.head()

,disease,symptoms
0,panic disorder,"anxiety and nervousness, shortness of breath, ..."
1,panic disorder,"shortness of breath, depressive or psychotic s..."
2,panic disorder,"anxiety and nervousness, depression, shortness..."
3,panic disorder,"anxiety and nervousness, depressive or psychot..."
4,panic disorder,"anxiety and nervousness, depression, insomnia,..."


In [ ]:
df['symptoms'][1]

'shortness of breath, depressive or psychotic symptoms, dizziness, insomnia, palpitations'

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246945 entries, 0 to 246944
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   disease   246945 non-null  object
 1   symptoms  246945 non-null  object
dtypes: object(2)
memory usage: 3.8+ MB


In [ ]:
df.isna().sum()

,0
disease,0
symptoms,0


In [ ]:
df.duplicated().sum()

np.int64(57298)

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.describe().T

,count,unique,top,freq
disease,189647,773,cystitis,1219
symptoms,189647,169888,sharp abdominal pain,26


In [ ]:
df['disease'].value_counts()

,count
disease,
cystitis,1219
nose disorder,1218
vulvodynia,1218
complex regional pain syndrome,1217
spondylosis,1216
...,...
kaposi sarcoma,1
open wound of the jaw,1
scurvy,1


In [ ]:
from sklearn.preprocessing import LabelEncoder

top_200 = (df['disease'].value_counts().head(100).index)

df = df[df['disease'].isin(top_200)].reset_index(drop=True)

encoder = LabelEncoder()

df['target'] = encoder.fit_transform(df['disease'])

df.head()

,disease,symptoms,target
0,panic disorder,"anxiety and nervousness, shortness of breath, ...",70
1,panic disorder,"shortness of breath, depressive or psychotic s...",70
2,panic disorder,"anxiety and nervousness, depression, shortness...",70
3,panic disorder,"anxiety and nervousness, depressive or psychot...",70
4,panic disorder,"anxiety and nervousness, depression, insomnia,...",70


In [ ]:
df['disease'].value_counts()

,count
disease,
cystitis,1219
vulvodynia,1218
nose disorder,1218
complex regional pain syndrome,1217
spondylosis,1216
...,...
hypertensive heart disease,804
gum disease,803
injury to the trunk,803


In [ ]:
import nltk
nltk.download('stopwords', quiet=True)

True

In [ ]:
import re
from nltk.corpus import stopwords

stop_words = stopwords.words('english')


def clean_text(text):

    text = str(text)

    text = text.lower()


    text = re.sub(r'[^a-z\s]',' ',text)


    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    words = []

    for word in text.split():

        if word not in stop_words:

            words.append(word)

    return " ".join(words)


df["clean_text"] = (
    df["symptoms"]
    .apply(clean_text)
)

df.head()

,disease,symptoms,target,clean_text
0,panic disorder,"anxiety and nervousness, shortness of breath, ...",70,anxiety nervousness shortness breath depressiv...
1,panic disorder,"shortness of breath, depressive or psychotic s...",70,shortness breath depressive psychotic symptoms...
2,panic disorder,"anxiety and nervousness, depression, shortness...",70,anxiety nervousness depression shortness breat...
3,panic disorder,"anxiety and nervousness, depressive or psychot...",70,anxiety nervousness depressive psychotic sympt...
4,panic disorder,"anxiety and nervousness, depression, insomnia,...",70,anxiety nervousness depression insomnia abnorm...


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(df['clean_text'])

In [ ]:
X = df['clean_text']
y = df['target']

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

In [ ]:
max_len = max(len(i) for i in X)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
X_train_pad = pad_sequences(X_train,maxlen=max_len,padding='pre')
X_test_pad = pad_sequences(X_test,maxlen=max_len,padding='pre')

In [ ]:
from keras.models import Sequential
from keras.layers import Embedding,Bidirectional,LSTM,Input,Dense

In [ ]:
model = Sequential()

model.add(Input(shape=(X_train_pad.shape[1],)))
model.add(Embedding(len(tokenizer.word_index)+1,100))
model.add(Bidirectional(LSTM(250,dropout=0.3)))
model.add(Dense(200,activation='softmax'))

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 242, 100)       │        25,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 500)            │       702,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 200)            │       100,200 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 827,300 (3.16 MB)

 Trainable params: 827,300 (3.16 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_accuracy',patience=5,restore_best_weights=True)

In [ ]:
history = model.fit(X_train_pad,y_train,validation_data=(X_test_pad, y_test),batch_size=256,epochs=3,callbacks=[early_stop])

Epoch 1/3
301/301 ━━━━━━━━━━━━━━━━━━━━ 48s 161ms/step - accuracy: 0.8531 - loss: 0.4053 - val_accuracy: 0.8737 - val_loss: 0.3174
Epoch 2/3
301/301 ━━━━━━━━━━━━━━━━━━━━ 47s 157ms/step - accuracy: 0.8733 - loss: 0.3247 - val_accuracy: 0.8766 - val_loss: 0.2884
Epoch 3/3
301/301 ━━━━━━━━━━━━━━━━━━━━ 48s 158ms/step - accuracy: 0.8788 - loss: 0.2933 - val_accuracy: 0.8818 - val_loss: 0.2643


In [ ]:
def predict_top3(text):

    seq = tokenizer.texts_to_sequences([text])

    seq = pad_sequences(seq,maxlen=X_train_pad.shape[1])

    pred = model.predict(seq)[0]

    top = pred.argsort()[-3:][::-1]

    for i in top:

        disease = encoder.inverse_transform([i])[0]

        print(f"{disease} → {pred[i]:.2%}")

In [ ]:
predict_top3('anxiety and nervousness, depression, depressive or psychotic symptoms, drug abuse, excessive anger, temper problems, fears and phobias, low self-esteem')

In [ ]:
from keras.models import Functional

In [ ]:
from keras.layers import Embedding
from keras.intializers import Kernal_

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

In [ ]:
import numpy as np

In [ ]:
y_pred = np.argmax(model.predict(X_test_pad), axis=1)

report = classification_report(y_test,y_pred,output_dict=True)

report_df = pd.DataFrame(report).transpose()

report_df[['precision','recall','f1-score','support']].tail(8)

601/601 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step


,precision,recall,f1-score,support
95,0.893130,0.657303,0.757282,178.00000
96,0.914530,0.775362,0.839216,138.00000
97,0.788779,0.888476,0.835664,269.00000
98,0.914439,0.988439,0.950000,173.00000
99,0.849624,0.941667,0.893281,240.00000
accuracy,0.881830,0.881830,0.881830,0.88183
macro avg,0.893744,0.881342,0.881204,19218.00000
weighted avg,0.895833,0.881830,0.882933,19218.00000


In [ ]:
model.save(r'Disease Prediction\disease_predictor.keras')

In [ ]:
import pickle

with open(r"tokenizer.pkl","wb") as f:
    pickle.dump(tokenizer,f)

In [ ]:
with open(r"Disease Prediction\label_encoder.pkl","wb") as f:
    pickle.dump(encoder,f)

In [ ]:
config = {"max_len": X_train_pad.shape[1]}

with open(r"Disease Prediction\config.pkl","wb") as f:
    pickle.dump(config,f)

In [ ]:
with open("preprocessor.pkl","wb") as f:
    pickle.dump(clean_text,f)